# Notebook 03 — Camada Silver

**MVP de Engenharia de Dados** · PUC-Rio · Sprint 3

---

### Objetivo

Converter os dados brutos em uma tabela tipada, padronizada e validada, pronta para ser modelada dimensionalmente na camada Gold.

### Entrada e saída

| | |
|---|---|
| **Entrada** | `workspace.bronze.balanco_energia_bruto` |
| **Saída** | `workspace.silver.balanco_energia` |

### A descoberta que motivou este notebook

A verificação de completude do notebook anterior retornou **43.800 registros por ano**, e não os 35.040 esperados. Dividindo, aparecem **5 registros por hora** em vez dos 4 subsistemas previstos.

O quinto é o **SIN — Sistema Interligado Nacional**, que não é um subsistema, e sim a **soma agregada dos outros quatro**. Confirmado pelos números: somando a geração hidráulica de SE, S, N e NE em qualquer instante, o resultado é exatamente o valor da linha SIN. O intercâmbio do SIN é zero, coerente com um sistema que não troca energia consigo mesmo.

Se esse registro fosse agregado junto com os demais, **todos os totais do trabalho sairiam dobrados** — um erro com aparência de acerto, que é o tipo mais perigoso.

O tratamento adotado preserva a rastreabilidade: a Silver **mantém todos os registros**, marcados por uma coluna indicadora, e a exclusão acontece apenas na camada Gold. Nenhum dado é descartado, e o SIN ainda será reaproveitado como referência independente para validar o próprio pipeline.

## 1. Tipagem e padronização

Quatro transformações e uma configuração.

**Tratamento do fuso horário.** O campo `din_instante` é publicado sem informação explícita de fuso. Embora a referência seja o horário de Brasília, a série apresenta exatamente 24 observações por dia durante todo o período, inclusive no início de 2019, quando ainda vigorava o horário de verão no Brasil. Isso indica que os valores devem ser interpretados como uma grade horária nominal, sem aplicação das transições históricas de horário de verão. Por esse motivo, a sessão Spark é fixada em UTC exclusivamente como convenção técnica, para impedir conversões automáticas e preservar exatamente a hora registrada no arquivo. Assim, `00:00` continua sendo analisado como hora 00 da série do ONS; o valor não deve ser interpretado como um instante UTC real. Uma evolução futura seria armazenar `din_instante` como `TIMESTAMP_NTZ`, tipo que representa diretamente data e hora sem semântica de fuso horário.

**`trim` nas colunas de texto, antes de tudo.** Qualquer regra que dependa do valor do texto, como a marcação do SIN, precisa ver o dado já sem espaços acidentais.

**Texto para `timestamp`.** Permite extrair ano, mês, hora e ordenar corretamente.

**Texto para `double`.** Permite agregação, e resolve a notação científica `0E-8`, que aparece na origem como representação de zero.

**Coluna `flag_agregado`.** Marca o registro SIN sem removê-lo, calculada sobre o texto já padronizado.

In [0]:
from pyspark.sql import functions as F
# Sessão em UTC como convenção técnica: impede conversões automáticas e preserva
# exatamente a hora do arquivo. O valor é a hora nominal da série do ONS, não um instante UTC.
spark.conf.set("spark.sql.session.timeZone", "UTC")

TABELA_BRONZE = "workspace.bronze.balanco_energia_bruto"
TABELA_SILVER = "workspace.silver.balanco_energia"

COLUNAS_NUMERICAS = [
    "val_gerhidraulica", "val_gertermica", "val_gereolica",
    "val_gersolar", "val_carga", "val_intercambio",
]

df = spark.table(TABELA_BRONZE)

# 1) Padroniza o texto primeiro: regras que dependem do valor devem ver o dado limpo
df = (df
      .withColumn("id_subsistema",  F.trim(F.col("id_subsistema")))
      .withColumn("nom_subsistema", F.trim(F.col("nom_subsistema"))))

# 2) Texto -> timestamp
df = df.withColumn("din_instante", F.to_timestamp("din_instante", "yyyy-MM-dd HH:mm:ss"))

# 3) Texto -> double (o '0E-8' é notação científica de zero e o cast resolve)
for coluna in COLUNAS_NUMERICAS:
    df = df.withColumn(coluna, F.col(coluna).cast("double"))

# 4) Marca a linha de total nacional, já sobre o texto padronizado
df = df.withColumn("flag_agregado", F.col("id_subsistema") == F.lit("SIN"))

df.printSchema()
display(df.limit(5))

root
 |-- id_subsistema: string (nullable = true)
 |-- nom_subsistema: string (nullable = true)
 |-- din_instante: timestamp (nullable = true)
 |-- val_gerhidraulica: double (nullable = true)
 |-- val_gertermica: double (nullable = true)
 |-- val_gereolica: double (nullable = true)
 |-- val_gersolar: double (nullable = true)
 |-- val_carga: double (nullable = true)
 |-- val_intercambio: double (nullable = true)
 |-- _arquivo_origem: string (nullable = true)
 |-- _data_ingestao: timestamp (nullable = true)
 |-- flag_agregado: boolean (nullable = true)



id_subsistema,nom_subsistema,din_instante,val_gerhidraulica,val_gertermica,val_gereolica,val_gersolar,val_carga,val_intercambio,_arquivo_origem,_data_ingestao,flag_agregado
NE,NORDESTE,2019-01-01T00:00:00.000Z,2292.417,873.482,5320.80899999,0.0,9831.71799999,-1345.01,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T01:46:48.593Z,false
N,NORTE,2019-01-01T00:00:00.000Z,7297.073,1416.71899999,142.237,0.0,4888.033,3967.996,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T01:46:48.593Z,false
SIN,SISTEMA INTERLIGADO NACIONAL,2019-01-01T00:00:00.000Z,41461.542,7826.763,6081.406,0.0,55369.71,0.0,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T01:46:48.593Z,true
SE,SUDESTE/CENTRO-OESTE,2019-01-01T00:00:00.000Z,28304.91799999,5007.999,0.0,0.0,31079.29999999,2233.61699999,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T01:46:48.593Z,false
S,SUL,2019-01-01T00:00:00.000Z,3567.13399999,528.563,618.36,0.0,9570.66099999,-4856.604,BALANCO_ENERGIA_SUBSISTEMA_2019.csv,2026-09-25T01:46:48.593Z,false


## 2. Verificação de unicidade

A chave de negócio é a combinação **subsistema + instante**: não pode existir duas leituras do mesmo subsistema na mesma hora.

Duplicatas silenciosas são um dos erros mais comuns em pipelines, porque não geram exceção — apenas inflam os totais.

In [0]:
total     = df.count()
distintos = df.select("id_subsistema", "din_instante").distinct().count()

print(f"Total de linhas      : {total:,}")
print(f"Chaves distintas     : {distintos:,}")
print(f"Duplicatas           : {total - distintos:,}")

Total de linhas      : 306,840
Chaves distintas     : 306,840
Duplicatas           : 0


## 3. Conformidade com o dicionário da fonte

As regras verificadas abaixo **não foram arbitradas**: são as especificações do dicionário de dados oficial do ONS.

| Campo | Regra documentada pela fonte |
|---|---|
| Colunas de geração | Aceitam nulo e zero, nunca valores negativos |
| `val_carga` | Não aceita nulo, aceita zero |
| `val_intercambio` | Único campo que admite valores negativos |

Validar contra a especificação de quem publicou o dado é mais defensável que validar contra o próprio palpite.

In [0]:
regras = {
    "hidraulica_negativa": F.col("val_gerhidraulica") < 0,
    "termica_negativa":    F.col("val_gertermica")    < 0,
    "eolica_negativa":     F.col("val_gereolica")     < 0,
    "solar_negativa":      F.col("val_gersolar")      < 0,
    "carga_nula":          F.col("val_carga").isNull(),
    "carga_negativa":      F.col("val_carga")         < 0,
    "instante_nulo":       F.col("din_instante").isNull(),
}

display(df.select([
    F.sum(F.when(cond, 1).otherwise(0)).alias(nome)
    for nome, cond in regras.items()
]))

hidraulica_negativa,termica_negativa,eolica_negativa,solar_negativa,carga_nula,carga_negativa,instante_nulo
0,0,0,0,0,0,0


### Verificação dos casts

As regras acima não detectam um tipo de falha: um valor de texto inválido na origem, por exemplo `"ERRO"`, que o cast não consegue converter. Dependendo da configuração do Spark, esse valor vira nulo sem gerar erro. Como as colunas de geração aceitam nulo pelo dicionário do ONS, ele passaria por todas as regras de domínio.

A célula abaixo compara a Bronze com a conversão e conta os valores que existiam no texto original e se perderam no cast. Usa `try_cast`, que se comporta da mesma forma em qualquer configuração. Qualquer resultado diferente de zero interrompe o pipeline antes da gravação da Silver.

In [0]:
bronze = spark.table(TABELA_BRONZE)

falhas = {}
for coluna in COLUNAS_NUMERICAS:
    falhas[coluna] = bronze.filter(
        F.col(coluna).isNotNull()
        & (F.trim(F.col(coluna)) != "")
        & F.expr(f"try_cast({coluna} AS DOUBLE)").isNull()
    ).count()

falhas["din_instante"] = bronze.filter(
    F.col("din_instante").isNotNull()
    & F.expr("try_to_timestamp(din_instante, 'yyyy-MM-dd HH:mm:ss')").isNull()
).count()

for coluna, n in falhas.items():
    print(f"{coluna:20s} valores perdidos no cast: {n}")

if sum(falhas.values()) > 0:
    raise ValueError("Há valores na Bronze que não puderam ser convertidos. Investigar antes de gravar a Silver.")

print("\nCasts OK: nenhum valor da origem foi perdido na conversão.")

val_gerhidraulica    valores perdidos no cast: 0
val_gertermica       valores perdidos no cast: 0
val_gereolica        valores perdidos no cast: 0
val_gersolar         valores perdidos no cast: 0
val_carga            valores perdidos no cast: 0
val_intercambio      valores perdidos no cast: 0
din_instante         valores perdidos no cast: 0

Casts OK: nenhum valor da origem foi perdido na conversão.


## 4. Persistência da camada Silver

Todos os 306.840 registros são gravados, incluindo os do SIN. A separação entre dado agregado e dado detalhado acontece na camada Gold, não aqui.

In [0]:
(df.write
   .mode("overwrite")
   .option("overwriteSchema", "true")
   .saveAsTable(TABELA_SILVER))

print(f"Tabela criada: {TABELA_SILVER}")
print(f"Linhas: {spark.table(TABELA_SILVER).count():,}")

Tabela criada: workspace.silver.balanco_energia
Linhas: 306,840
